# 🛒 Superstore Sales — Exploratory Data Analysis

**数据集**：`superstore_sales.xlsx`，零售超市销售数据  
**分析维度**：销售趋势 · 品类表现 · 区域分布 · 客户分层（RFM） · 相关性分析 · 时间序列预测（Prophet）

---

## 1. 导入依赖

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Mac 系统中文字体设置
matplotlib.rcParams['font.family'] = 'Arial Unicode MS'
matplotlib.rcParams['axes.unicode_minus'] = False

%matplotlib inline

## 2. 数据加载与清洗

读取原始数据，统一日期格式，检查缺失值和重复行。

In [ ]:
data = pd.read_excel('superstore_sales.xlsx')

# 统一日期格式并排序
data['order_date'] = pd.to_datetime(data['order_date'])
data = data.sort_values('order_date').reset_index(drop=True)

# 添加年月字段（用于后续按月聚合）
data['sale_month'] = data['order_date'].apply(lambda x: x.strftime('%Y-%m'))

# 检查数据质量
print('=== 缺失值 ===')
print(data.isnull().sum())
print(f'\n重复行数: {data.duplicated().sum()}')
data = data.drop_duplicates()
print(f'数据范围: {data["order_date"].min().date()} 至 {data["order_date"].max().date()}')

In [ ]:
data.head()

In [ ]:
data.info()

In [ ]:
data.describe()

## 3. 销售趋势分析

通过月度趋势、年度季节性对比和滚动均值，了解整体销售走势和周期规律。

In [ ]:
# 按月汇总总销售额（resample 需要 DatetimeIndex）
monthly_sales = (
    data.set_index('order_date')['sales']
    .resample('MS')          # 'MS' = 每月第一天
    .sum()
    .rename('total_sales')
)

print(monthly_sales.head(12))
print(f'\n共 {len(monthly_sales)} 个月的数据')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# 图1：月度销售趋势
axes[0].plot(monthly_sales.index, monthly_sales.values, linewidth=1.5, color='steelblue')
axes[0].set_title('月度销售总额趋势', fontsize=14)
axes[0].set_ylabel('销售额')
axes[0].grid(True, alpha=0.3)

# 图2：各年度月度对比（每年一条线，揭示季节性规律）
monthly_sales_df = monthly_sales.reset_index()
monthly_sales_df['year']  = monthly_sales_df['order_date'].dt.year
monthly_sales_df['month'] = monthly_sales_df['order_date'].dt.month

for year, group in monthly_sales_df.groupby('year'):
    axes[1].plot(group['month'], group['total_sales'], marker='o', label=str(year))
axes[1].set_title('各年度月度对比（揭示季节性）', fontsize=14)
axes[1].set_xlabel('月份')
axes[1].set_xticks(range(1, 13))
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 图3：12 个月滚动均值（平滑短期波动，显示长期趋势）
rolling_avg = monthly_sales.rolling(window=12, center=True).mean()
axes[2].plot(monthly_sales.index, monthly_sales.values, alpha=0.4, label='原始数据')
axes[2].plot(rolling_avg.index, rolling_avg.values, linewidth=2.5, color='red', label='12月滚动均值')
axes[2].set_title('趋势平滑', fontsize=14)
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 各品类月度销售趋势
category_trend = data.groupby(['sale_month', 'category'])['sales'].sum().reset_index()

plt.figure(figsize=(16, 8))
sns.lineplot(data=category_trend, x='sale_month', y='sales', hue='category', marker='o')
plt.title('Monthly Sales Trend by Category', fontsize=15)
plt.xticks(rotation=45)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(title='Category')
plt.show()

In [ ]:
# 销售额 vs 平均折扣力度（双轴图）
sales_discount_trend = data.groupby('sale_month').agg({'sales': 'sum', 'discount': 'mean'}).reset_index()

fig, ax1 = plt.subplots(figsize=(16, 8))

ax1.set_xlabel('Month')
ax1.set_ylabel('Total Sales', color='navy', fontsize=12)
sns.lineplot(data=sales_discount_trend, x='sale_month', y='sales',
             ax=ax1, alpha=0.3, color='navy', label='Total Sales')
ax1.tick_params(axis='y', labelcolor='navy')

ax2 = ax1.twinx()
ax2.set_ylabel('Average Discount', color='darkorange', fontsize=12)
sns.lineplot(data=sales_discount_trend, x='sale_month', y='discount',
             ax=ax2, color='darkorange', marker='o', linewidth=2, label='Avg Discount')
ax2.tick_params(axis='y', labelcolor='darkorange')

plt.title('Relationship between Sales Volume and Discount Intensity', fontsize=16)
ax1.set_xticklabels(sales_discount_trend['sale_month'], rotation=45)

lines, labels   = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax2.legend(lines + lines2, labels + labels2, loc='upper left')

plt.show()

In [ ]:
# 同比增速（YoY）分析
monthly_sales_yoy = data.groupby('sale_month')['sales'].sum().reset_index()
monthly_sales_yoy['last_year_sales'] = monthly_sales_yoy['sales'].shift(12)  # 对应去年同月
monthly_sales_yoy['yoy_growth'] = (
    (monthly_sales_yoy['sales'] - monthly_sales_yoy['last_year_sales'])
    / monthly_sales_yoy['last_year_sales']
) * 100

# 第一年无去年同期数据，剔除 NaN
yoy_data = monthly_sales_yoy.dropna(subset=['yoy_growth'])

plt.figure(figsize=(14, 6))
sns.lineplot(data=yoy_data, x='sale_month', y='yoy_growth', marker='o', color='teal', linewidth=2)
plt.axhline(0, color='red', linestyle='--', alpha=0.5)  # 0 基准线：正负增长分界
plt.title('Monthly Sales Year-over-Year (YoY) Growth Rate', fontsize=15)
plt.ylabel('Growth Rate (%)')
plt.xlabel('Month')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)

for i in range(len(yoy_data)):
    plt.text(yoy_data.iloc[i]['sale_month'],
             yoy_data.iloc[i]['yoy_growth'] + 1,
             f"{yoy_data.iloc[i]['yoy_growth']:.1f}%",
             ha='center', fontsize=9)
plt.show()

## 4. 时间序列分解与平稳性检验

使用 STL 分解将时间序列拆解为趋势、季节性和残差三部分，并通过 ADF 检验判断序列是否平稳（Prophet 建模的前提检查）。

In [ ]:
# ADF 检验：对一阶差分序列判断平稳性
monthly_sales_diff = monthly_sales.diff().dropna()
result = adfuller(monthly_sales_diff)

print(f'ADF 统计量: {result[0]:.4f}')
print(f'p 值      : {result[1]:.4f}')
print('结论:', '序列平稳 ✓' if result[1] < 0.05 else '序列不平稳，需要差分')

In [ ]:
# STL 分解：拆解趋势 + 季节 + 残差
decomp = seasonal_decompose(monthly_sales, model='multiplicative', period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomp.observed.plot(ax=axes[0], title='原始序列')
decomp.trend.plot(ax=axes[1],    title='趋势（Trend）')
decomp.seasonal.plot(ax=axes[2], title='季节性（Seasonality）')
decomp.resid.plot(ax=axes[3],    title='残差（Residual）')
plt.tight_layout()
plt.show()

In [ ]:
# 对比 additive 和 multiplicative 两种分解模型的残差
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

for i, model_type in enumerate(['additive', 'multiplicative']):
    decomp = seasonal_decompose(monthly_sales, model=model_type, period=12)
    axes[i].plot(decomp.resid)
    axes[i].set_title(f'{model_type} 残差')
    axes[i].axhline(0, color='red', linewidth=0.8, linestyle='--')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. 销售预测（Prophet）

使用 Meta 开源的 Prophet 模型预测未来 12 个月的销售额，并分别对三个大品类（Furniture / Office Supplies / Technology）单独建模。

In [ ]:
# Prophet 要求列名为 'ds'（日期）和 'y'（目标值）
df_prophet = monthly_sales.reset_index()
df_prophet.columns = ['ds', 'y']

model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    seasonality_mode='multiplicative'  # 季节波动随趋势增长时用 multiplicative
)
model.fit(df_prophet)

# 预测未来 12 个月
future   = model.make_future_dataframe(periods=12, freq='MS')
forecast = model.predict(future)

fig1 = model.plot(forecast)
plt.title('Prophet 预测：未来12个月销售额')
plt.xlabel('日期')
plt.ylabel('销售额')
plt.show()

# 趋势和季节性分解图
fig2 = model.plot_components(forecast)
plt.show()

print(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(12))

In [ ]:
# 用最后 12 个月做验证集，评估模型误差
train = df_prophet.iloc[:-12]
test  = df_prophet.iloc[-12:]

model_eval = Prophet(yearly_seasonality=True, seasonality_mode='multiplicative')
model_eval.fit(train)

future_eval   = model_eval.make_future_dataframe(periods=12, freq='MS')
forecast_eval = model_eval.predict(future_eval)
pred   = forecast_eval['yhat'].tail(12).values
actual = test['y'].values

mae  = mean_absolute_error(actual, pred)
rmse = np.sqrt(mean_squared_error(actual, pred))
mape = np.mean(np.abs((actual - pred) / actual)) * 100

print(f'MAE  (平均绝对误差)  : {mae:,.0f}')
print(f'RMSE (均方根误差)    : {rmse:,.0f}')
print(f'MAPE (平均百分比误差): {mape:.1f}%')

In [ ]:
# 可视化：真实值 vs 预测值
pred_lower = forecast_eval['yhat_lower'].tail(12).values
pred_upper = forecast_eval['yhat_upper'].tail(12).values

plt.figure(figsize=(14, 5))
plt.plot(train['ds'], train['y'],
         label='历史数据（训练集）', color='steelblue', linewidth=1.5)
plt.plot(test['ds'], actual,
         label='真实值（测试集）', color='black', linewidth=2, marker='o', markersize=5)
plt.plot(test['ds'], pred,
         label='模型预测值', color='tomato', linewidth=2, linestyle='--', marker='x', markersize=6)
plt.fill_between(test['ds'], pred_lower, pred_upper,
                 alpha=0.15, color='tomato', label='预测区间')
plt.axvline(x=test['ds'].iloc[0], color='gray',
            linestyle=':', linewidth=1.5, label='训练/测试切割点')
plt.title(f'模型评估：预测值 vs 真实值   (MAPE={mape:.1f}%)', fontsize=13)
plt.xlabel('日期')
plt.ylabel('月度销售额')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def forecast_category(sales_series, category_name, periods=12):
    """
    对单个品类的月度销售序列进行 Prophet 建模：
    1. 用最后 12 个月做验证，输出 MAE / RMSE / MAPE
    2. 用全量数据重新训练，预测真正的未来 periods 个月
    """
    df = sales_series.reset_index()
    df.columns = ['ds', 'y']

    train = df.iloc[:-12]
    test  = df.iloc[-12:]

    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        seasonality_mode='multiplicative'
    )
    model.fit(train)

    future_eval   = model.make_future_dataframe(periods=12, freq='MS')
    forecast_eval = model.predict(future_eval)
    pred   = forecast_eval['yhat'].tail(12).values
    actual = test['y'].values

    mae  = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mape = np.mean(np.abs((actual - pred) / actual)) * 100

    print(f"\n{'='*40}")
    print(f'类别：{category_name}')
    print(f'MAE  : {mae:>12,.0f} 元')
    print(f'RMSE : {rmse:>12,.0f} 元')
    print(f'MAPE : {mape:>11.1f} %')

    # 用全量数据预测真正的未来
    model_full = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        seasonality_mode='multiplicative'
    )
    model_full.fit(df)
    future_full   = model_full.make_future_dataframe(periods=periods, freq='MS')
    forecast_full = model_full.predict(future_full)

    result = forecast_full[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(periods).copy()
    result.columns = ['月份', '预测销售额', '下限', '上限']
    result['月份'] = result['月份'].dt.strftime('%Y-%m')
    print(f'\n未来 {periods} 个月预测：')
    print(result.to_string(index=False))

    fig, axes = plt.subplots(2, 1, figsize=(14, 8))

    axes[0].plot(train['ds'], train['y'], color='steelblue', label='训练集', linewidth=1.5)
    axes[0].plot(test['ds'], actual, color='black', marker='o', linewidth=2, label='真实值')
    axes[0].plot(test['ds'], pred, color='tomato', linestyle='--', marker='x',
                 linewidth=2, label=f'预测值 (MAPE={mape:.1f}%)')
    axes[0].fill_between(test['ds'],
                         forecast_eval['yhat_lower'].tail(12).values,
                         forecast_eval['yhat_upper'].tail(12).values,
                         alpha=0.15, color='tomato')
    axes[0].axvline(x=test['ds'].iloc[0], color='gray', linestyle=':', linewidth=1.5)
    axes[0].set_title(f'{category_name}：模型评估')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    future_dates = forecast_full['ds'].tail(periods)
    future_yhat  = forecast_full['yhat'].tail(periods)
    future_lower = forecast_full['yhat_lower'].tail(periods)
    future_upper = forecast_full['yhat_upper'].tail(periods)

    axes[1].plot(df['ds'], df['y'], color='steelblue', label='历史数据', linewidth=1.5)
    axes[1].plot(future_dates, future_yhat, color='tomato', linestyle='--',
                 marker='o', linewidth=2, label='未来预测')
    axes[1].fill_between(future_dates, future_lower, future_upper,
                         alpha=0.2, color='tomato', label='预测区间')
    axes[1].axvline(x=df['ds'].iloc[-1], color='gray',
                    linestyle=':', linewidth=1.5, label='历史/未来分界')
    axes[1].set_title(f'{category_name}：未来 {periods} 个月需求预测')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    return forecast_full


# 对三个大类分别建模预测
for cat in ['Furniture', 'Office Supplies', 'Technology']:
    cat_sales = (
        data[data['category'] == cat]
        .set_index('order_date')['sales']
        .resample('MS')
        .sum()
    )
    forecast_category(cat_sales, cat)

## 6. 品类与区域分析

分析不同产品品类和子品类的销售额与利润分布，以及各区域的表现差异。

In [ ]:
# 各品类子品类利润汇总
category_best_profit = (
    data.groupby(['category', 'sub_category'])['profit']
    .sum()
    .reset_index()
    .sort_values(['category', 'profit'], ascending=False)
)
category_best_profit

In [ ]:
# 子品类销售额占比饼图（小于 5% 的合并为 Other）
sub_category_sales = data.groupby('sub_category')['sales'].sum().sort_values(ascending=False)
total_sales        = sub_category_sales.sum()
percentages        = sub_category_sales / total_sales

main_categories       = sub_category_sales[percentages >= 0.05]
others                = pd.Series({'Other': sub_category_sales[percentages < 0.05].sum()})
category_sales_merged = pd.concat([main_categories, others])

plt.figure(figsize=(8, 8))
plt.pie(category_sales_merged, labels=category_sales_merged.index,
        autopct='%1.1f%%', startangle=140,
        colors=sns.color_palette('Set3'), wedgeprops={'edgecolor': 'black'})
plt.title('Sales Distribution by Sub-Category')
plt.show()

In [ ]:
# 大品类销售额占比饼图
category_sales = data.groupby('category')['sales'].sum().reset_index()

plt.figure(figsize=(8, 8))
plt.pie(category_sales['sales'], labels=category_sales['category'],
        autopct='%1.1f%%', startangle=140,
        colors=sns.color_palette('Set3'), wedgeprops={'edgecolor': 'black'})
plt.title('Sales Distribution by Category')
plt.show()

In [ ]:
# 区域 × 品类销售额热力图
pivot_table = data.pivot_table(index='region', columns='category', values='sales', aggfunc='sum')

plt.figure(figsize=(8, 5))
sns.heatmap(pivot_table, annot=True, cmap='YlGnBu', fmt='.0f')
plt.title('Sales by Region and Category')
plt.show()

In [ ]:
# 各区域订单数量分布
plt.figure(figsize=(10, 5))
sns.countplot(x='region', data=data, palette='Set3')
plt.title('Order Count by Region')
plt.show()

## 7. 产品分析

找出销售额和销量 Top 10 的产品，以及物流配送方式分布。

In [ ]:
# Top 10 销售额产品
product_trend       = data.groupby('product_name')['sales'].sum().reset_index()
top10_sale_products = product_trend.sort_values(by='sales', ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x='sales', y='product_name', data=top10_sale_products, palette='viridis')
plt.title('Top 10 Products by Sales')
plt.xlabel('Total Sales')
plt.ylabel('Product Name')
plt.show()

In [ ]:
# Top 10 销量产品
top10_quantity_products = (
    data.groupby('product_name')['quantity'].sum()
    .reset_index()
    .sort_values(by='quantity', ascending=False)
    .head(10)
)
top10_quantity_products

In [ ]:
# 物流配送方式分布
plt.figure(figsize=(10, 5))
sns.countplot(x=data['ship_mode'], palette='Set2')
plt.title('Order Count by Ship Mode')
plt.show()

## 8. 客户分析（RFM 分层）

使用 RFM 模型（Recency 最近购买时间 / Frequency 购买频次 / Monetary 消费金额）对客户进行分层，识别高价值客户和流失风险客户。

In [ ]:
# Top 10 客户（销售额 + 购买频次）
customer_sales  = data.groupby('customer_name')['sales'].sum().sort_values(ascending=False)
customer_orders = data['customer_name'].value_counts()

customer_stats = pd.DataFrame({
    'total_sales': customer_sales,
    'order_count': customer_orders
}).sort_values('total_sales', ascending=False)

print(customer_stats.head(10))

In [ ]:
# 计算 RFM 三个指标（以数据最晚日期作为参考点）
today = data['order_date'].max()

rfm = data.groupby('customer_name').agg(
    recency  =('order_date', lambda x: (today - x.max()).days),  # 距最近购买天数
    frequency=('order_id', 'count'),                             # 购买次数
    monetary =('sales', 'sum')                                   # 总消费额
)

print(rfm.head())

In [ ]:
# 四分位打分并进行客户分层
rfm['r_score'] = pd.qcut(rfm['recency'], 4, labels=['4', '3', '2', '1'])   # 越近分越高
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 4, labels=['1', '2', '3', '4'])
rfm['m_score'] = pd.qcut(rfm['monetary'].rank(method='first'), 4, labels=['1', '2', '3', '4'])

rfm['rfm_score']   = rfm['r_score'].astype(str) + rfm['f_score'].astype(str) + rfm['m_score'].astype(str)
rfm['total_score'] = rfm['r_score'].astype(int) + rfm['f_score'].astype(int) + rfm['m_score'].astype(int)

def segment_customer(row):
    if row['total_score'] >= 10:  return 'ssvip'
    elif row['total_score'] >= 8: return 'svip'
    elif row['total_score'] >= 6: return 'vip'
    elif row['frequency'] == 1:   return 'fresh'
    elif row['recency'] > 180:    return 'sleeping'
    else:                         return 'normal'

rfm['segment'] = rfm.apply(segment_customer, axis=1)
print(rfm['segment'].value_counts())

In [ ]:
# 客户分层占比（饼图 + 条形图）
segment_counts = rfm['segment'].value_counts()
colors = ['#FFD700', '#C0C0C0', '#CD7F32', '#87CEEB', '#A9A9A9', '#98FB98']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].pie(segment_counts.values, labels=segment_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, explode=[0.05]*len(segment_counts))
axes[0].set_title('Customer Segmentation', fontsize=14, fontweight='bold')

bars = axes[1].barh(segment_counts.index, segment_counts.values, color=colors)
axes[1].set_xlabel('Customer Count')
axes[1].set_title('Number of Customers in Each Segment')
for bar, val in zip(bars, segment_counts.values):
    axes[1].text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2, f'{val}', va='center')

plt.tight_layout()
plt.show()

In [ ]:
# 各层级客户 RFM 平均表现热力图
segment_rfm = rfm.groupby('segment')[['recency', 'frequency', 'monetary']].mean().round(2)

plt.figure(figsize=(10, 6))
sns.heatmap(segment_rfm, annot=True, fmt='.0f', cmap='RdYlGn', center=segment_rfm.values.mean())
plt.title('RFM Profile by Customer Segment')
plt.tight_layout()
plt.show()

In [ ]:
# 购买频次 vs 消费金额散点图（颜色代表 RFM 综合得分）
plt.figure(figsize=(12, 8))
scatter = plt.scatter(rfm['frequency'], rfm['monetary'],
                      c=rfm['total_score'], cmap='viridis',
                      alpha=0.6, s=rfm['frequency']*10)
plt.colorbar(scatter, label='RFM Score')
plt.xlabel('Frequency', fontsize=12)
plt.ylabel('Monetary', fontsize=12)
plt.title('Customer Purchase Behavior Matrix: Frequency vs Amount', fontsize=14, fontweight='bold')

# 用中位数分割四个象限
plt.axhline(y=rfm['monetary'].median(),  color='red', linestyle='--', alpha=0.5)
plt.axvline(x=rfm['frequency'].median(), color='red', linestyle='--', alpha=0.5)

plt.text(rfm['frequency'].max()*0.7, rfm['monetary'].max()*0.9,  'High-Value Customers', fontsize=12)
plt.text(rfm['frequency'].min()*2,   rfm['monetary'].max()*0.9,  'Potential Customers', fontsize=12)
plt.text(rfm['frequency'].max()*0.7, rfm['monetary'].min()*5,    'Loyal but Low-Spending', fontsize=12)
plt.text(rfm['frequency'].min()*2,   rfm['monetary'].min()*5,    'Low-Value Customers', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# 各层级客户品类偏好热力图
customer_segment  = rfm['segment'].to_frame()
data_with_segment = data.merge(customer_segment, on='customer_name')

segment_category = pd.crosstab(
    data_with_segment['segment_y'],
    data_with_segment['category'],
    values=data_with_segment['sales'],
    aggfunc='sum', normalize='index'
) * 100

plt.figure(figsize=(12, 6))
sns.heatmap(segment_category, annot=True, fmt='.1f', cmap='Blues',
            cbar_kws={'label': '销售额占比(%)'})
plt.title('各层级客户品类偏好热力图', fontsize=14, fontweight='bold')
plt.ylabel('客户层级')
plt.xlabel('产品类别')
plt.tight_layout()
plt.show()

## 9. 相关性分析

分析销售额、折扣、利润、数量和运费之间的相关关系，重点关注折扣对利润的影响。

In [ ]:
numeric_cols = ['sales', 'quantity', 'discount', 'profit', 'shipping_cost']
df_num = data[numeric_cols].copy()

print('=== Descriptive Statistics ===')
print(df_num.describe())

In [ ]:
# 相关性热力图
corr_matrix = df_num.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5,
            cbar_kws={'label': 'Correlation Coefficient'})
plt.title('Correlation Heatmap: Sales, Discount, Profit & Shipping Cost', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 散点图矩阵（多变量关系总览）
sns.pairplot(df_num, diag_kind='kde', plot_kws={'alpha': 0.5})
plt.suptitle('Scatter Plot Matrix of Numerical Variables', y=1.02, fontsize=14)
plt.show()

In [ ]:
# 折扣 vs 利润（最关键的关系）
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 散点图：折扣率 vs 利润
sns.scatterplot(x='discount', y='profit', data=data, alpha=0.5, color='coral', ax=axes[0])
axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.7)  # 亏损分界线
axes[0].set_title('Discount vs Profit\n(Negative values indicate loss)', fontsize=12)
axes[0].set_xlabel('Discount Rate')
axes[0].set_ylabel('Profit')

# 柱状图：按折扣区间统计平均利润
discount_bins      = pd.cut(data['discount'], bins=10)
profit_by_discount = data.groupby(discount_bins)['profit'].mean()
profit_by_discount.plot(kind='bar', color='steelblue', ax=axes[1])
axes[1].set_title('Average Profit by Discount Interval', fontsize=12)
axes[1].set_xlabel('Discount Interval')
axes[1].set_ylabel('Average Profit')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# 销售额 vs 利润（颜色代表折扣率）
plt.figure(figsize=(10, 6))
scatter = plt.scatter(data['sales'], data['profit'],
                      c=data['discount'], cmap='coolwarm', alpha=0.5)
plt.colorbar(scatter, label='Discount Rate')
plt.axhline(y=0, color='red', linestyle='--', alpha=0.7)
plt.xlabel('Sales')
plt.ylabel('Profit')
plt.title('Sales vs Profit (Color = Discount Rate)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# 销售数量 vs 销售额（箱线图，限 quantity <= 10）
plt.figure(figsize=(8, 5))
sns.boxplot(x='quantity', y='sales', data=data[data['quantity'] <= 10])
plt.title('Sales Distribution by Quantity Purchased')
plt.xlabel('Quantity')
plt.ylabel('Sales')
plt.tight_layout()
plt.show()

In [ ]:
# 按品类拆分折扣 vs 利润关系
if 'category' in data.columns:
    g = sns.FacetGrid(data, col='category', col_wrap=3, height=4)
    g.map(sns.scatterplot, 'discount', 'profit', alpha=0.5)
    g.set_axis_labels('Discount', 'Profit')
    g.fig.subplots_adjust(top=0.9)
    g.fig.suptitle('Discount vs Profit by Product Category')
    plt.show()

In [ ]:
# Pearson 相关系数显著性检验，结果保存为 CSV
report = []
for col in ['sales', 'profit']:
    for var in ['discount', 'quantity', 'shipping_cost']:
        r, p = stats.pearsonr(df_num[col], df_num[var])
        report.append({
            'Y变量': col,
            'X变量': var,
            '相关系数': round(r, 3),
            'P值': round(p, 4),
            '显著性': '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '不显著'
        })

corr_result = pd.DataFrame(report)
corr_result.to_csv('相关性分析结果.csv', index=False)
print(corr_result.to_string(index=False))
print("\n结果已保存为 '相关性分析结果.csv'")